# Lab | Tools prompting

**Replace the existing two tools decorators, by creating 3 new ones and adjust the prompts accordingly**

### How to add ad-hoc tool calling capability to LLMs and Chat Models

:::{.callout-caution}

Some models have been fine-tuned for tool calling and provide a dedicated API for tool calling. Generally, such models are better at tool calling than non-fine-tuned models, and are recommended for use cases that require tool calling. Please see the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide for more information.

In this guide, we'll see how to add **ad-hoc** tool calling support to a chat model. This is an alternative method to invoke tools if you're using a model that does not natively support tool calling.

We'll do this by simply writing a prompt that will get the model to invoke the appropriate tools. Here's a diagram of the logic:

<br>

![chain](https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/tool_chain.svg)

## Setup

We'll need to install the following packages:

In [1]:
%pip install --upgrade --quiet langchain langchain-community langchain_openai

Note: you may need to restart the kernel to use updated packages.


If you'd like to use LangSmith, uncomment the below:

In [2]:
import getpass
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()

You can select any of the given models for this how-to guide. Keep in mind that most of these models already [support native tool calling](https://python.langchain.com/docs/integrations/chat), so using the prompting strategy shown here doesn't make sense for these models, and instead you should follow the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide.

```{=mdx}
import ChatModelTabs from "@theme/ChatModelTabs";

<ChatModelTabs openaiParams={`model="gpt-4"`} />
```

To illustrate the idea, we'll use `phi3` via Ollama, which does **NOT** have native support for tool calling. If you'd like to use `Ollama` as well follow [these instructions](https://python.langchain.com/docs/integrations/chat/ollama).

In [3]:
!pip install langchain-community langchain_openai

In [10]:
# Install the new package because of deprecation warning
%pip install -U langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-ollama]
Note: you may need to restart the kernel to use updated packages.


In [11]:
# New import because of previous deprecation warning
from langchain_ollama import OllamaLLM

model = OllamaLLM(model="phi3")


#  How to Install and Run Ollama with the Phi-3 Model

This guide walks you through installing **Ollama** and running the **Phi-3** model on Windows, macOS, and Linux.

---

## Windows

1. **Download Ollama for Windows**  
   Go to: [https://ollama.com/download](https://ollama.com/download)  
   Download and run the installer.

2. **Verify Installation**  
   Open **Command Prompt** and type:
   ```bash
   ollama --version
   ```

3. **Run the Phi-3 Model**  
   In the same terminal:
   ```bash
   ollama run phi3
   ```

4. **If you get a CUDA error (GPU memory issue)**  
   Run Ollama in **CPU mode**:
   ```bash
   set OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  macOS

1. **Install via Homebrew**  
   Open the Terminal and run:
   ```bash
   brew install ollama
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  Linux

1. **Install Ollama**  
   Open a terminal and run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  Notes

- The first time you run `ollama run phi3`, it will **download the model**, so make sure you’re connected to the internet.
- Once downloaded, it works **offline**.
- Keep the terminal open and running in the background while using Ollama from your code or notebook.


## Create a tool

First, let's create an `add` and `multiply` tools. For more information on creating custom tools, please see [this guide](https://python.langchain.com/docs/how_to/custom_tools/).

In [22]:
from langchain_core.tools import tool


@tool
def calculate_bmi(weight:float, height:float) -> float:
    """"Calculate the BMI of a person, using weight in kilograms and height in centimeters."""
    return weight / (height ** 2)


@tool
def calculate_discount(price:float, percent:float) -> float:
    """"Calculate the final price of an an article, after a percentage discount."""
    return price - (price * percent / 100)

@tool
def convert_celsius_to_fahrenheit(celsius: float) ->float:
    """"Convert Celsius temperature to Fahrenheit"""
    return (celsius * 9/5) + 32


tools = [
    calculate_bmi,
    calculate_discount,
    convert_celsius_to_fahrenheit
    ]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
calculate_bmi
"Calculate the BMI of a person, using weight in kilograms and height in centimeters.
{'weight': {'title': 'Weight', 'type': 'number'}, 'height': {'title': 'Height', 'type': 'number'}}
--
calculate_discount
"Calculate the final price of an an article, after a percentage discount.
{'price': {'title': 'Price', 'type': 'number'}, 'percent': {'title': 'Percent', 'type': 'number'}}
--
convert_celsius_to_fahrenheit
"Convert Celsius temperature to Fahrenheit
{'celsius': {'title': 'Celsius', 'type': 'number'}}


In [29]:
calculate_bmi.invoke({"weight": 85, "height": 183})

0.0025381468541909283

In [31]:
calculate_discount.invoke({"price": 45, "percent": 10})

40.5

In [33]:
convert_celsius_to_fahrenheit.invoke({"celsius": 30})

86.0

## Creating our prompt

We'll want to write a prompt that specifies the tools the model has access to, the arguments to those tools, and the desired output format of the model. In this case we'll instruct it to output a JSON blob of the form `{"name": "...", "arguments": {...}}`.

In [34]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools) #this is a utility function in Langchain to convert tools into text description
print(rendered_tools)

calculate_bmi(weight: float, height: float) -> float - "Calculate the BMI of a person, using weight in kilograms and height in centimeters.
calculate_discount(price: float, percent: float) -> float - "Calculate the final price of an an article, after a percentage discount.
convert_celsius_to_fahrenheit(celsius: float) -> float - "Convert Celsius temperature to Fahrenheit


In [36]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools. 
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use. 
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding 
to the argument names and the values corresponding to the requested values.
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

In [37]:
chain = prompt | model
message = chain.invoke({"input": "How many Fahrenheit would equal to 25 degrees Celsius?"})
# Let's take a look at the output from the model
# if the model is an LLM (not a chat model), the output will be a string.
if isinstance(message, str):
    print(message)
else:  # Otherwise it's a chat model
    print(message.content)

```json
{
  "name": "convert_celsius_to_fahrenheit",
  "arguments": {
    "celsius": 25
  }
}
```



## Adding an output parser

We'll use the `JsonOutputParser` for parsing our models output to JSON.

In [38]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | model | JsonOutputParser()
chain.invoke({"input": "how much money will I be able to save if I buy a car that costs 43.567,13€ and has a discout of 27 peercent"})

{'name': 'calculate_discount', 'arguments': {'price': 43.56713, 'percent': 27}}

:::{.callout-important}

🎉 Amazing! 🎉 We now instructed our model on how to **request** that a tool be invoked.

Now, let's create some logic to actually run the tool!
:::

## Invoking the tool 🏃

Now that the model can request that a tool be invoked, we need to write a function that can actually invoke 
the tool.

The function will select the appropriate tool by name, and pass to it the arguments chosen by the model.

In [40]:
from typing import Any, Dict, Optional, TypedDict

from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""

    name: str
    arguments: Dict[str, Any]


def invoke_tool(
    tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None
):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

Let's test this out 🧪!

In [41]:
invoke_tool({"name": "calculate_bmi", "arguments": {"weight": 74, "height": 189}})

0.002071610537219003

## Let's put it together

Let's put it together into a chain that creates a calculator with add and multiplication capabilities.

In [42]:
chain = prompt | model | JsonOutputParser() | invoke_tool
chain.invoke({"input": "How high is the BMI of a person that is weighing 123,42kg and is 213 centimeters tall?"})

0.0027203597169873704

## Returning tool inputs

It can be helpful to return not only tool outputs but also tool inputs. We can easily do this with LCEL by `RunnablePassthrough.assign`-ing the tool output. This will take whatever the input is to the RunnablePassrthrough components (assumed to be a dictionary) and add a key to it while still passing through everything that's currently in the input:

In [43]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)
chain.invoke({"input": "How high is the BMI of a person that is weighing 123,42kg and is 213 centimeters tall?"})

{'name': 'calculate_bmi',
 'arguments': {'weight': 123.42, 'height': 213},
 'output': 0.0027203597169873704}

In [49]:
chain.invoke({"input": "How many degrees Fahrenheit are 239,7 degrees Celsius?"})

{'name': 'convert_celsius_to_fahrenheit',
 'arguments': {'celsius': 239.7},
 'output': 463.4599999999999}

In [47]:
chain.invoke({"input": "How much money will I have to pay if I buy 24 products that cost 17.64€ and have 8.73 percent discount?"})

{'name': 'calculate_discount',
 'arguments': {'price': 24, 'percent': 8.73},
 'output': 21.9048}

## What's next?

This how-to guide shows the "happy path" when the model correctly outputs all the required tool information.

In reality, if you're using more complex tools, you will start encountering errors from the model, especially for models that have not been fine tuned for tool calling and for less capable models.

You will need to be prepared to add strategies to improve the output from the model; e.g.,

1. Provide few shot examples.
2. Add error handling (e.g., catch the exception and feed it back to the LLM to ask it to correct its previous output).

# Improved Prompt with Few-Shot Examples

In [ ]:
# Improved Prompt with Few-Shot Examples

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

system_prompt = f"""\
You are an assistant that has access to the following tools.

Tools:
{rendered_tools}

You must return ONLY valid JSON.
Do not add explanations.
Do not use markdown.
Do not wrap the JSON in ```json.

Return this exact structure:
{{{{"name": "tool_name", "arguments": {{{{"x": number, "y": number}}}}}}}}

Examples:

User input: what is 3 plus 5
Response:
{{{{"name": "add", "arguments": {{{{"x": 3, "y": 5}}}}}}}}

User input: what is 4 times 7
Response:
{{{{"name": "multiply", "arguments": {{{{"x": 4, "y": 7}}}}}}}}

User input: multiply 10 by 2
Response:
{{{{"name": "multiply", "arguments": {{{{"x": 10, "y": 2}}}}}}}}

Given the user input, choose the correct tool and arguments.
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("user", "{input}")
    ]
)

chain = prompt | model | JsonOutputParser()

chain.invoke({"input": "what is thirteen times 4"})

{'name': 'multiply', 'arguments': {'x': 13, 'y': 4}}

# Error Handling for Invalid Tool Calls

In [ ]:
from typing import Any, Dict
import json

def safe_invoke_tool(tool_call_request: Dict[str, Any]):
    try:
        tool_name_to_tool = {tool.name: tool for tool in tools}

        name = tool_call_request["name"]
        arguments = tool_call_request["arguments"]

        requested_tool = tool_name_to_tool[name]

        return requested_tool.invoke(arguments)

    except Exception as e:
        return {
            "error": str(e),
            "original_output": tool_call_request
        }

In [ ]:
#Test it
safe_invoke_tool({
    "name": "multiply",
    "arguments": {
        "x": 5,
        "y": 4
    }
})

20.0

In [ ]:
#Break it
safe_invoke_tool({
    "name": "multiply",
    "arguments": {
        "a": 5,
        "b": 4
    }
})

{'error': "2 validation errors for multiply\nx\n  Field required [type=missing, input_value={'a': 5, 'b': 4}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing\ny\n  Field required [type=missing, input_value={'a': 5, 'b': 4}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing",
 'original_output': {'name': 'multiply', 'arguments': {'a': 5, 'b': 4}}}

In [ ]:
#Correcting Invalid Tool Calls with the LLM

def fix_tool_call(error_message, original_output):
    correction_prompt = f"""
The previous tool call was invalid.

Error:
{error_message}

Original tool call:
{original_output}

You must fix it using this exact JSON schema:
{{"name": "multiply", "arguments": {{"x": 5, "y": 4}}}}

Rules:
- Return ONLY JSON
- Use lowercase tool name
- Use keys: name and arguments
- arguments must contain x and y directly
- Do not add Result
- Do not add Value
- Do not add explanation
"""

    response = model.invoke(correction_prompt)

    return response

In [ ]:
bad_tool_call = {
    "name": "multiply",
    "arguments": {
        "a": 5,
        "b": 4
    }
}

result = safe_invoke_tool(bad_tool_call)

fixed_response = fix_tool_call(
    result["error"],
    result["original_output"]
)

print(fixed_response)

{"name": "multiply", "arguments": {"x": 5, "y": 4}}



In [ ]:
# Try validating the corrected output again

fixed_output = {
    "name": "multiply",
    "arguments": {
        "x": 5,
        "y": 4
    }
}

validated = multiply.args_schema(**fixed_output["arguments"])

print(validated)

x=5.0 y=4.0


In [ ]:
final_result = safe_invoke_tool(fixed_output)

print(final_result)

20.0
